# Cyrus, Winnowed

In [1]:
from winnow_fcns import *
from sim_utils import *
import numpy as np
from pathlib import Path
import os
import csv

In [2]:
# class MockBitBuffer:
#     def __init__(self, bits):
#         self.bits = list(np.array(bits).astype(int))
#         self.seed = None
#     def get_length(self): return len(self.bits)
#     def get_bit(self, i): return self.bits[i]
#     def set_bit(self, i): self.bits[i] = 1
#     def clear_bit(self, i): self.bits[i] = 0
#     def flip_bit(self, i): self.bits[i] = 1 - self.bits[i]
#     def set_seed(self, s): self.seed = s
#     # def permute_buffer(self): 
#     def permute_buffer(self):
#         """
#         Shuffles the bits based on the seed. 
#         Alice and Bob use the same seed to ensure identical shuffles.
#         """
#         if self.seed is None:
#             raise ValueError("Seed must be set before permutation!")
        
#         # We use a local generator so we don't mess up other random streams
#         rng = np.random.default_rng(self.seed)
#         rng.shuffle(self.bits)

In [3]:
# rng = np.random.default_rng(seed=42)
# rng2 = np.random.default_rng(seed=180)
# N = 30
# std = 0.1
# ber = 0.25


# init_key = rng2.integers(low=0, high=2, size=N)


# print(init_key)
# key = simulate_error(init_key, rng, ber=ber, N=N)
# print(key)


#------------------files---------------------------------------------
current_file = Path.cwd()

init_file = current_file.parent / "initial_cyrus_bits.csv"
ch1_file = current_file.parent / "ch1_cyrus.csv"
ch2_file = current_file.parent / "ch2_cyrus.csv"



def load_data(init_file, ch1_file, ch2_file):
    init = np.loadtxt(init_file, delimiter=',', dtype=int)
    ch1 = np.loadtxt(ch1_file, delimiter=',')
    ch2 = np.loadtxt(ch2_file, delimiter=',')
    return init, ch1, ch2


init_key, ch1, ch2 = load_data(init_file, ch1_file, ch2_file)


In [5]:
alice_key = MockBitBuffer(list(ch1))
bob_key   = MockBitBuffer(list(ch2))


#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

# start first pass
alice_winnow.first_pass(permute_bits=True)
bob_winnow.first_pass(permute_bits=True)

# alice calculates syndrome for the first block 
alice_syndrome = alice_winnow.get_syndrome(0)

# bob uses Alice's syndrome to fix his key
print(f"Bob's key before fix: {bob_winnow._key_string.bits[:8]}")
bob_winnow.fix_with_syndrome(0, alice_syndrome)
print(f"Bob's key after fix:  {bob_winnow._key_string.bits[:8]}")
print(f"Alice's key: {alice_winnow._key_string.bits[:8]}")

# verify that the keys match
if alice_winnow._key_string.bits[:8] == bob_winnow._key_string.bits[:8]:
    print(" Success! Bob corrected the error.")
else:
    print("Failure: Keys still do not match.")



Bob's key before fix: [np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
Bob's key after fix:  [np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
Alice's key: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
Failure: Keys still do not match.


In [ ]:
alice_key = MockBitBuffer(list(ch1))
bob_key   = MockBitBuffer(list(ch2))


#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

# start first pass
alice_winnow.first_pass()
bob_winnow.first_pass()

success_sample = 0
total_sample = 0
with open('corrected/cyrus_test.csv', 'w', newline='') as f:
    # alice calculates syndrome for the first block 
    for i in range(alice_winnow._num_of_blocks):
        block_size = alice_winnow._block_size
        startb = i*block_size
        endb = block_size*(i + 1) 
        alice_syndrome = alice_winnow.get_syndrome(i)
        # bob uses Alice's syndrome to fix his key
        if i % 1000 == 0:
            print(f"Bob's key before fix: {bob_key.bits[startb:endb]} block {i}")
        bob_winnow.fix_with_syndrome(i, alice_syndrome)
        if i % 1000 == 0:
            print(f"Bob's key after fix:  {bob_key.bits[startb:endb]} block {i}")
            print(f"Alice's key: {alice_key.bits[startb:endb]}")

        # verify that the keys match
        if i % 100 == 0:
            
            if alice_key.bits[startb:endb] == bob_key.bits[startb:endb]:
                print(f" Success! Bob corrected the error for block {i}.")
                success_sample+=1
            else:
                print(f"Failure: Keys still do not match for block {i}.")
            total_sample+=1
        
        writer = csv.writer(f)
        for item in bob_key.bits[:8]:
            writer.writerow([item]) # Wrap item in a list

if alice_key.bits == bob_key.bits:
    print(" Success! Bob corrected the error.")
else:
    print("Failure: Keys still do not match.")

print(f"Percentage success {(success_sample/total_sample )* 100} percent")

Bob's key before fix: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0)] block 0
Bob's key after fix:  [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)] block 0
Alice's key: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
 Success! Bob corrected the error for block 0.
Failure: Keys still do not match for block 100.
Failure: Keys still do not match for block 200.
Failure: Keys still do not match for block 300.
Failure: Keys still do not match for block 400.
Failure: Keys still do not match for block 500.
Failure: Keys still do not match for block 600.
Failure: Keys still do not match for block 700.
Failure: Keys still do not match for block 800.
Failure: Keys still do not match for block 900.
Bob's key before fix: [np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0)] block 

In [55]:
matches = (np.array(alice_key.bits)  == np.array(bob_key.bits) )

# Calculate percentage
percentage = matches.mean() * 100

print(f"{percentage}%")

74.73812866210938%


# Lets apply to many blocks!

In [56]:
# print(alice_winnow._num_of_blocks)
# print(alice_winnow._block_size_schedule)
# alice_winnow.next_pass(permute_bits=True) 
# bob_winnow.next_pass(permute_bits=True) 
# print(alice_winnow._block_size_schedule)
# print(alice_winnow._num_of_blocks)

In [4]:
alice_key = MockBitBuffer(list(ch1))
bob_key   = MockBitBuffer(list(ch2))


#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

# start first pass
alice_winnow.first_pass()
bob_winnow.first_pass()

success_sample = 0
total_sample = 0
with open('corrected/cyrus_test.csv', 'w', newline='') as f:
    # alice calculates syndrome for the first block 
    for i in range(alice_winnow._num_of_blocks):
        block_size = alice_winnow._block_size
        startb = i*block_size
        endb = block_size*(i + 1) 
        alice_syndrome = alice_winnow.get_syndrome(i)
        # bob uses Alice's syndrome to fix his key
        # if i % 1000 == 0:
        #     print(f"Bob's key before fix: {bob_winnow._key_string.bits[startb:endb]} block {i}")
        bob_winnow.fix_with_syndrome(i, alice_syndrome)
        # if i % 1000 == 0:
        #     print(f"Bob's key after fix:  {bob_key.bits[startb:endb]} block {i}")
        #     print(f"Alice's key: {alice_key.bits[startb:endb]}")

        # # verify that the keys match
        # if i % 100 == 0:
            
        #     if alice_key.bits[startb:endb] == bob_key.bits[startb:endb]:
        #         print(f" Success! Bob corrected the error for block {i}.")
        #         success_sample+=1
        #     else:
        #         print(f"Failure: Keys still do not match for block {i}.")
        #     total_sample+=1
        
        # writer = csv.writer(f)
        # for item in bob_key.bits[:8]:
        #     writer.writerow([item]) # Wrap item in a list
            

# if alice_winnow._key_string.bits == bob_winnow._key_string.bits:
#     print(" Success! Bob corrected the error.")
# else:
#     print("Failure: Keys still do not match.")

# print(f"Percentage success {(success_sample/total_sample )* 100} percent")

matches = (np.array(alice_winnow._key_string.bits)  != np.array(bob_winnow._key_string.bits) )

# Calculate percentage
percentage = matches.mean() * 100

print(f"Qber is {percentage}%")

while alice_winnow.get_num_remaining_passes() > 0:
    # print("\nAlice internal:", alice_winnow._key_string.bits[:20])
    # print("Bob   internal:", bob_winnow._key_string.bits[:20])

    print(f"Alice buffer length: {len(alice_winnow._key_string.bits)}")
    print(f"Bob   buffer length: {len(bob_winnow._key_string.bits)}")
    print(f"Alice seed: {alice_winnow._key_string.seed}")
    print(f"Bob seed:   {bob_winnow._key_string.seed}") 
    block_size  = alice_winnow._block_size
    num_blocks  = alice_winnow._num_of_blocks
    alice_winnow._key_string.discard_parity_bits(block_size, num_blocks)
    bob_winnow._key_string.discard_parity_bits(block_size, num_blocks)
    alice_winnow.next_pass(permute_bits=True) 
    bob_winnow.next_pass(permute_bits=True) 
    # print("Alice internal:", alice_winnow._key_string.bits[:20])
    # print("Bob   internal:", bob_winnow._key_string.bits[:20])
    # print(f"Alice pass number {alice_winnow._pass_number}")
    # print(f"Bob ass number {bob_winnow._pass_number}")
    # with open('corrected/cyrus_test.csv', 'w', newline='') as f:
    # # alice calculates syndrome for the first block 
    for i in range(alice_winnow._num_of_blocks):
        block_size = alice_winnow._block_size
        startb = i*block_size
        endb = block_size*(i + 1) 
        alice_syndrome = alice_winnow.get_syndrome(i)
        # bob uses Alice's syndrome to fix his key
        # if i % 1000 == 0:
        #     print(f"Bob's key before fix: {bob_key.bits[startb:endb]} block {i}")
        bob_winnow.fix_with_syndrome(i, alice_syndrome)
        # if i % 1000 == 0:
        #     print(f"Bob's key after fix:  {bob_key.bits[startb:endb]} block {i}")
        #     print(f"Alice's key: {alice_key.bits[startb:endb]}")

        # # verify that the keys match
        # if i % 100 == 0:
            
        #     if alice_key.bits[startb:endb] == bob_key.bits[startb:endb]:
        #         print(f" Success! Bob corrected the error for block {i}.")
        #         # success_sample+=1
        #     else:
        #         print(f"Failure: Keys still do not match for block {i}.")
        #     # total_sample+=1
        
        # writer = csv.writer(f)
        # for item in bob_key.bits[:8]:
        #     writer.writerow([item]) # Wrap item in a list

    matches = (np.array(alice_winnow._key_string.bits)  != np.array(bob_winnow._key_string.bits) )

    # Calculate percentage
    percentage = matches.mean() * 100

    print(f"Qber is {percentage}%")
    print(alice_winnow.num_remaining_errors(bob_winnow._key_string))



Qber is 28.432312011718754%
Alice buffer length: 13107200
Bob   buffer length: 13107200
Alice seed: 42
Bob seed:   42
Qber is 26.329267229352677%
3019651
Alice buffer length: 11468800
Bob   buffer length: 11468800
Alice seed: 44
Bob seed:   44
Qber is 27.007264429209183%
2710233
Alice buffer length: 10035200
Bob   buffer length: 10035200
Alice seed: 47
Bob seed:   47
Qber is 25.6790041909621%
2254822
Alice buffer length: 8780800
Bob   buffer length: 8780800
Alice seed: 51
Bob seed:   51
Qber is 25.58921282798834%
2106504


In [ ]:
# after all winnow passes and discards are done
alice_winnow._key_string.unpermute_all()
bob_winnow._key_string.unpermute_all()

# verify they still match after unscrambling
a = np.array(alice_winnow._key_string.bits)
b = np.array(bob_winnow._key_string.bits)
print(f"QBER after unscrambling: {(a != b).mean() * 100:.4f}%")

In [ ]:
# # success_sample = 0
# # total_sample = 0
# with open('corrected/cyrus_test.csv', 'w', newline='') as f:
#     # alice calculates syndrome for the first block 
#     for i in range(alice_winnow._num_of_blocks):
#         block_size = alice_winnow._block_size
#         startb = i*block_size
#         endb = block_size*(i + 1) 
#         alice_syndrome = alice_winnow.get_syndrome(i)
#         # bob uses Alice's syndrome to fix his key
#         if i % 1000 == 0:
#             print(f"Bob's key before fix: {bob_key.bits[startb:endb]} block {i}")
#         bob_winnow.fix_with_syndrome(i, alice_syndrome)
#         if i % 1000 == 0:
#             print(f"Bob's key after fix:  {bob_key.bits[startb:endb]} block {i}")
#             print(f"Alice's key: {alice_key.bits[startb:endb]}")

#         # verify that the keys match
#         if i % 100 == 0:
            
#             if alice_key.bits[startb:endb] == bob_key.bits[startb:endb]:
#                 print(f" Success! Bob corrected the error for block {i}.")
#                 # success_sample+=1
#             else:
#                 print(f"Failure: Keys still do not match for block {i}.")
#             # total_sample+=1
        
#         writer = csv.writer(f)
#         for item in bob_key.bits[:8]:
#             writer.writerow([item]) # Wrap item in a list



# matches = (np.array(alice_key.bits)  == np.array(bob_key.bits) )

# # Calculate percentage
# percentage = matches.mean() * 100

# print(f"Qber is {percentage}%")

# if alice_key.bits == bob_key.bits:
#     print(" Success! Bob corrected the error.")
# else:
#     print("Failure: Keys still do not match.")

Bob's key before fix: [np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0)] block 0
Bob's key after fix:  [np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0)] block 0
Alice's key: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
Failure: Keys still do not match for block 0.
Failure: Keys still do not match for block 100.
 Success! Bob corrected the error for block 200.
Failure: Keys still do not match for block 300.
Failure: Keys still do not match for block 400.
Failure: Keys still do not match for block 500.
Failure: Keys still do not match for block 600.
Failure: Keys still do not match for block 700.
Failure: Keys still do not match for block 800.
Failure: Keys still do not match for block 900.
Bob's key before fix: [np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1)] block 

In [31]:
matches = (np.array(alice_key.bits)  == np.array(bob_key.bits) )

# Calculate percentage
percentage = matches.mean() * 100

print(f"{percentage}%")

73.64989471435547%


In [32]:
print(f"Percentage success {(success_sample/total_sample )* 100} percent")

Percentage success 32.04345703125 percent


Lets do multiple passes!